# 01 · Clasificación binaria con  TensorFlow/Keras (actualizado)

## Objetivo
Entender, paso a paso, cómo una red neuronal aprende a separar datos en **dos clases** (0 y 1).

Este notebook es un estudio propio, inspirado en los talleres de clase, pero con **datos distintos** y **teoría intercalada** en cada paso para entender el "por qué", no solo el "cómo".

## Teoría: ¿qué es clasificación binaria?

Es un problema de Machine Learning donde el modelo debe responder con **una de dos opciones posibles**: sí/no, 0/1, verdadero/falso.

Ejemplos reales: un correo es spam o no es spam; un paciente tiene o no tiene cierta condición; una transacción es fraudulenta o no.

Aquí usaremos un ejemplo simple y controlado: clasificar números como **negativos (clase 0)** o **positivos (clase 1)**. Es artificial a propósito, para poder concentrarnos en entender la arquitectura de la red sin la complejidad de un dataset real.


## Paso 1: Importar las librerías

- **TensorFlow**: la biblioteca de Deep Learning que usaremos para construir y entrenar la red neuronal.
- **NumPy**: nos permite crear y manipular arreglos numéricos (matrices), que es el formato que TensorFlow espera como entrada.


In [2]:
import tensorflow as tf
import numpy as np

print("Versión de TensorFlow:", tf.__version__)

Versión de TensorFlow: 2.21.0


## Paso 2: Preparar los datos

En todo problema supervisado necesitamos dos cosas:
- **X (entradas)**: los datos que el modelo va a observar.
- **Y (etiquetas)**: la respuesta correcta para cada dato de X.

Nota importante: en Keras, X normalmente debe tener forma de **matriz de columna**, por eso cada valor va dentro de su propio corchete `[valor]`. Aquí usamos números del -10 al 10, saltando el 0, para que la separación entre clases sea clara.


In [3]:
X = np.array([
    [-10], [-8], [-6], [-4], [-2],
    [2], [4], [6], [8], [10]
])

# Etiquetas: 0 para negativos, 1 para positivos
Y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

print("X:\n", X)
print("Y:", Y)

X:
 [[-10]
 [ -8]
 [ -6]
 [ -4]
 [ -2]
 [  2]
 [  4]
 [  6]
 [  8]
 [ 10]]
Y: [0 0 0 0 0 1 1 1 1 1]


## Paso 3: Construir la arquitectura de la red

Aquí es donde conectamos la teoría del **perceptrón** con el código:

- `tf.keras.Sequential([...])`: define un modelo como una **secuencia de capas**, una detrás de otra.
- `Dense(8, activation="relu", input_shape=(1,))`: es la **capa oculta**. "Dense" significa que cada neurona de esta capa está conectada a todas las entradas. Tiene 8 neuronas, cada una calculando su propio `Z = ΣXᵢWᵢ + b` y aplicando la función de activación **ReLU** (`f(x) = max(0, x)`).
- `input_shape=(1,)`: le decimos a la red que cada dato de entrada tiene 1 solo valor (un número).
- `Dense(1, activation="sigmoid")`: es la **capa de salida**. Una sola neurona con activación **sigmoid**, que convierte el resultado en un número entre 0 y 1 — interpretable como una probabilidad de pertenecer a la clase 1.

En resumen: 1 entrada → 8 neuronas ocultas (ReLU) → 1 neurona de salida (sigmoid).


In [4]:
modelo = tf.keras.models.Sequential([
    tf.keras.layers.Dense(8, activation="relu", input_shape=(1,)),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

modelo.summary()

c:\Users\ANT DOR\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25 (100.00 B)

 Trainable params: 25 (100.00 B)

 Non-trainable params: 0 (0.00 B)

## Paso 4: Compilar el modelo

"Compilar" significa configurar **cómo va a aprender** el modelo, antes de entrenarlo:

- **optimizer="adam"**: el algoritmo que ajusta los pesos y el bias en cada paso, buscando minimizar el error. Adam ajusta automáticamente qué tan grandes son esos ajustes (a diferencia de una tasa de aprendizaje fija).
- **loss="binary_crossentropy"**: la función de pérdida adecuada quand el problema es de clasificación binaria (2 clases). Mide qué tan lejos está la probabilidad predicha de la etiqueta real (0 o 1).


In [5]:
modelo.compile(optimizer="adam", loss="binary_crossentropy")

## Paso 5: Entrenar el modelo

`.fit(X, Y, epochs=100)` inicia el entrenamiento:

- Una **época (epoch)** es una pasada completa por todos los datos de entrenamiento.
- En cada época, el modelo predice, compara con la etiqueta real, calcula el error, y ajusta los pesos (esto es la regla de aprendizaje que vimos con el perceptrón, pero aplicada a las 8 neuronas de la capa oculta y la de salida).
- Con 100 épocas, el modelo ve los mismos 10 datos 100 veces, mejorando un poco en cada vuelta.


In [6]:
historial = modelo.fit(X, Y, epochs=100, verbose=0)
print("Pérdida final:", historial.history["loss"][-1])

Pérdida final: 0.12449371814727783


## Paso 6: Predecir con datos nuevos

Ahora probamos el modelo con un número que **nunca vio durante el entrenamiento**.

- `modelo.predict(...)` devuelve la probabilidad (salida de la sigmoid, entre 0 y 1).
- `np.round(...)` la redondea a 0 o 1 para obtener la clase final.


In [7]:
nuevo_dato = np.array([[-50]])
probabilidad = modelo.predict(nuevo_dato)

print("Probabilidad de ser clase 1:", probabilidad)
print("Clase predicha (redondeada):", np.round(probabilidad))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Probabilidad de ser clase 1: [[1.040376e-21]]
Clase predicha (redondeada): [[0.]]


## Conclusión del notebook

- Vimos el flujo completo: preparar datos → definir arquitectura → compilar → entrenar → predecir.
- La capa oculta con ReLU le da a la red la capacidad de aprender un límite de decisión (no solo una línea recta).
- La salida sigmoid es lo que hace que este sea un problema de **clasificación binaria** y no de regresión.
- Todo esto es la misma lógica del perceptrón, pero apilada en varias neuronas y capas.
